# Reflexion Architecture — Google ADK

**Pattern:** `score_recommendation` tool triggers self-critique

```
Draft (LLM)
    │
    ▼
score_recommendation() ──score < 7──▶ Rewrite (LLM)
    │                                       │
 score >= 7                          score_recommendation()
    │
    ▼
  Output
```

**ADK approach:** Reflexion is embedded in the instruction. The `score_recommendation` tool is the evaluator. The loop is LLM-controlled — same as ADK 03-complex but now shown as a standalone architecture.

In [ ]:
import asyncio, os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
print("✓ Ready")

In [ ]:
def score_recommendation(draft: str) -> dict:
    """Evaluate quality of a travel recommendation.
    Call after drafting. If score < 7, rewrite addressing suggestions.
    Args:
        draft: The recommendation text.
    Returns:
        Dict with score, pass, and suggestions.
    """
    score = 0; suggestions = []
    if any(k in draft.lower() for k in ["attraction","temple","museum","market","park"]): score += 3
    else: suggestions.append("Add specific named attractions.")
    if any(k in draft.lower() for k in ["°c","weather","temperature","season","rain"]): score += 2
    else: suggestions.append("Include weather/temperature.")
    if any(k in draft.lower() for k in ["safety","safe","advisory","precaution"]): score += 2
    else: suggestions.append("Add safety tips.")
    if len(draft) > 400: score += 3
    elif len(draft) > 200: score += 1; suggestions.append("Expand detail.")
    else: suggestions.append("Too brief.")
    return {"score": min(score,10), "threshold": 7, "pass": score>=7, "suggestions": suggestions or ["Looks great!"]}

print("Evaluator demo:")
print("Bad draft:",  score_recommendation("Tokyo is nice."))
print("Good draft:", score_recommendation("Tokyo has Senso-ji temple, 18°C clear weather. Safety: Low advisory. " * 5))

In [ ]:
agent = Agent(
    name="reflexion_writer", model="gemini-2.0-flash",
    description="Writes travel recommendations with built-in self-critique.",
    instruction="""STEP 1: Write a travel recommendation (attractions, weather, safety, best time).\nSTEP 2: Call score_recommendation() with your draft.\n  - score >= 7 → deliver final answer\n  - score < 7  → rewrite addressing suggestions, call score_recommendation() again\nMUST call score_recommendation() at least once.""",
    tools=[score_recommendation])
print(f"Agent: {agent.name}")

In [ ]:
async def run(destination):
    ss = InMemorySessionService()
    session = await ss.create_session(app_name="reflexion_writer", user_id="u1")
    runner = InMemoryRunner(agent=agent, session_service=ss)
    final = ""; calls = 0
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=f"Write a recommendation for {destination}.")])): 
        if hasattr(event,"tool_call") and event.tool_call:
            calls += 1; print(f"  [score_recommendation] call #{calls}")
        elif hasattr(event,"tool_result") and event.tool_result:
            s = str(event.tool_result)
            if "score" in s: print(f"  → {s[:60]}")
        elif event.is_final_response() and event.content:
            for p in event.content.parts:
                if p.text: final += p.text
    return final, calls

result, calls = await run("Tokyo")
print(f"\n{calls} evaluation call(s)")
print(f"Final quality: {score_recommendation(result)['score']}/10")
print("\n--- Final Recommendation ---")
print(result)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `score_recommendation` as a tool | ADK reflexion = tool-based evaluation, no separate agent |
| LLM decides when to stop | Loop is LLM-controlled — flexible but less predictable |
| Multiple tool calls visible | Streaming shows each evaluation round |

### Reflexion: All 4 Frameworks Compared
| Framework | Loop Control | Evaluator | Max Retries |
|---|---|---|---|
| LangChain | Python `for` loop | Critic chain (regex parse) | `range(MAX_RETRIES)` |
| LangGraph | Conditional edge | Critic node | State counter |
| CrewAI | Fixed 3-task pipeline | Critic agent | Always 1 revision |
| ADK | LLM instruction | `score_recommendation` tool | LLM decides |

**Which to use?**
- LangGraph: most control + observable loop
- LangChain: simplest code
- CrewAI: if you always want one revision pass
- ADK: if you want the simplest single-agent setup